<a href="https://colab.research.google.com/github/dystaSatria/Deep-Learning/blob/main/Internship%20Projects/F0score/F0score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import OrderedDict

class DenseLayer(nn.Module):
    """Lapisan Dense Block untuk DenseNet"""
    def __init__(self, num_input_features, growth_rate, bn_size, drop_rate):
        super(DenseLayer, self).__init__()
        self.add_module('norm1', nn.BatchNorm2d(num_input_features))
        self.add_module('relu1', nn.ReLU(inplace=True))
        self.add_module('conv1', nn.Conv2d(num_input_features, bn_size * growth_rate,
                                         kernel_size=1, stride=1, bias=False))
        self.add_module('norm2', nn.BatchNorm2d(bn_size * growth_rate))
        self.add_module('relu2', nn.ReLU(inplace=True))
        self.add_module('conv2', nn.Conv2d(bn_size * growth_rate, growth_rate,
                                         kernel_size=3, stride=1, padding=1, bias=False))
        self.drop_rate = drop_rate

    def forward(self, x):
        new_features = self.conv1(self.relu1(self.norm1(x)))
        new_features = self.conv2(self.relu2(self.norm2(new_features)))
        if self.drop_rate > 0:
            new_features = F.dropout(new_features, p=self.drop_rate, training=self.training)
        return torch.cat([x, new_features], 1)

class DenseBlock(nn.Module):
    """Dense Block yang terdiri dari multiple Dense Layers"""
    def __init__(self, num_layers, num_input_features, bn_size, growth_rate, drop_rate):
        super(DenseBlock, self).__init__()
        for i in range(num_layers):
            layer = DenseLayer(num_input_features + i * growth_rate, growth_rate,
                             bn_size, drop_rate)
            self.add_module(f'denselayer{i+1}', layer)

    def forward(self, x):
        for name, layer in self.named_children():
            x = layer(x)
        return x

class Transition(nn.Module):
    """Transition layer untuk mengurangi dimensi feature maps"""
    def __init__(self, num_input_features, num_output_features):
        super(Transition, self).__init__()
        self.add_module('norm', nn.BatchNorm2d(num_input_features))
        self.add_module('relu', nn.ReLU(inplace=True))
        self.add_module('conv', nn.Conv2d(num_input_features, num_output_features,
                                        kernel_size=1, stride=1, bias=False))
        self.add_module('pool', nn.AvgPool2d(kernel_size=2, stride=2))

    def forward(self, x):
        return self.pool(self.conv(self.relu(self.norm(x))))

class DenseNet(nn.Module):
    """
    DenseNet Implementation dengan layer F0 (layer konvolusi awal)

    Args:
        growth_rate (int): Jumlah filter yang ditambahkan setiap layer
        block_config (tuple): Jumlah layer di setiap dense block
        num_init_features (int): Jumlah feature maps di layer F0
        bn_size (int): Multiplicative factor untuk bottleneck layers
        drop_rate (float): Dropout rate
        num_classes (int): Jumlah kelas output
    """

    def __init__(self, growth_rate=32, block_config=(6, 12, 24, 16),
                 num_init_features=64, bn_size=4, drop_rate=0, num_classes=1000):
        super(DenseNet, self).__init__()

        # Layer F0 - Layer konvolusi dan pooling awal
        self.features = nn.Sequential(OrderedDict([
            ('conv0', nn.Conv2d(3, num_init_features, kernel_size=7, stride=2,
                               padding=3, bias=False)),
            ('norm0', nn.BatchNorm2d(num_init_features)),
            ('relu0', nn.ReLU(inplace=True)),
            ('pool0', nn.MaxPool2d(kernel_size=3, stride=2, padding=1)),
        ]))

        # Dense blocks dan transition layers
        num_features = num_init_features
        for i, num_layers in enumerate(block_config):
            block = DenseBlock(num_layers, num_features, bn_size, growth_rate, drop_rate)
            self.features.add_module(f'denseblock{i+1}', block)
            num_features = num_features + num_layers * growth_rate

            if i != len(block_config) - 1:
                trans = Transition(num_features, num_features // 2)
                self.features.add_module(f'transition{i+1}', trans)
                num_features = num_features // 2

        # Final batch norm
        self.features.add_module('norm5', nn.BatchNorm2d(num_features))

        # Linear layer untuk klasifikasi
        self.classifier = nn.Linear(num_features, num_classes)

        # Weight initialization
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        features = self.features(x)
        out = F.relu(features, inplace=True)
        out = F.adaptive_avg_pool2d(out, (1, 1)).view(features.size(0), -1)
        out = self.classifier(out)
        return out

def densenet121(**kwargs):
    """DenseNet-121 model"""
    return DenseNet(growth_rate=32, block_config=(6, 12, 24, 16),
                    num_init_features=64, **kwargs)

def densenet169(**kwargs):
    """DenseNet-169 model"""
    return DenseNet(growth_rate=32, block_config=(6, 12, 32, 32),
                    num_init_features=64, **kwargs)

def densenet201(**kwargs):
    """DenseNet-201 model"""
    return DenseNet(growth_rate=32, block_config=(6, 12, 48, 32),
                    num_init_features=64, **kwargs)

# Contoh penggunaan
if __name__ == "__main__":
    # Buat model DenseNet-121
    model = densenet121(num_classes=10)

    # Test dengan input dummy
    x = torch.randn(2, 3, 224, 224)  # Batch size 2, 3 channel, 224x224

    print("Input shape:", x.shape)

    # Forward pass
    with torch.no_grad():
        output = model(x)
        print("Output shape:", output.shape)

    # Print model structure
    print("\nModel architecture:")
    print(model)

    # Hitung parameter
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nTotal parameters: {total_params:,}")

    # Test layer F0 secara terpisah
    print("\nTesting F0 layer (initial conv + pooling):")
    f0_features = model.features.conv0(x)
    print(f"After conv0: {f0_features.shape}")

    f0_features = model.features.norm0(f0_features)
    f0_features = model.features.relu0(f0_features)
    print(f"After norm0 + relu0: {f0_features.shape}")

    f0_features = model.features.pool0(f0_features)
    print(f"After pool0 (F0 complete): {f0_features.shape}")

Input shape: torch.Size([2, 3, 224, 224])
Output shape: torch.Size([2, 10])

Model architecture:
DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): DenseBlock(
      (denselayer1): DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): DenseLayer(
        (norm1):